# 03. 모델 학습 (Modeling)
**Credit Risk Dataset — 대출 금리 예측 프로젝트**

---
### 학습 단계
1. 전처리된 데이터 로드 및 분할
2. 4가지 회귀 모델 학습 비교 (선형·릿지·GBR·RF)
3. 분류 모델 학습 (RandomForest — 부도 예측)
4. 최종 모델 저장

---
## 0. 환경 설정

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

ROOT           = os.path.abspath('..')
PROCESSED_PATH = os.path.join(ROOT, 'data', 'processed', 'credit_risk_cleaned.csv')
MODEL_DIR      = os.path.join(ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
print('설정 완료')

---
## 1. 데이터 로드 및 분할

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
print(f'전처리 데이터: {df.shape[0]:,}행 x {df.shape[1]}컬럼')
df.head(3)

In [ ]:
FEATURES = [
    'person_age', 'person_income', 'person_home_ownership', 'person_emp_length',
    'loan_intent', 'loan_grade', 'loan_amnt', 'loan_status',
    'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length',
]
TARGET = 'loan_int_rate'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'학습: {len(X_train):,}행  /  테스트: {len(X_test):,}행')

---
## 2. 회귀 모델 학습 비교

In [ ]:
reg_models = {
    '선형 회귀':        LinearRegression(),
    '릿지 회귀':        Ridge(alpha=1.0),
    '그래디언트 부스팅': GradientBoostingRegressor(n_estimators=200, random_state=SEED),
    '랜덤 포레스트':    RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
}

results = {}
preds   = {}

for name, model in reg_models.items():
    print(f'{name} 학습 중...', end=' ')
    model.fit(X_train, y_train)
    y_pred        = model.predict(X_test)
    preds[name]   = y_pred
    rmse          = np.sqrt(mean_squared_error(y_test, y_pred))
    mae           = mean_absolute_error(y_test, y_pred)
    r2            = r2_score(y_test, y_pred)
    results[name] = {'RMSE': round(rmse,4), 'MAE': round(mae,4), 'R2': round(r2,4)}
    print(f'완료  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')

In [ ]:
df_res = pd.DataFrame(results).T
df_res.style.highlight_min(subset=['RMSE','MAE'], color='#c6efce') \
            .highlight_max(subset=['R2'],          color='#c6efce')

In [ ]:
model_names = list(results.keys())
colors      = ['#4C72B0','#55A868','#C44E52','#8172B2']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, label, lower in zip(
    axes,
    ['RMSE','MAE','R2'],
    ['RMSE (낮을수록 좋음)','MAE (낮을수록 좋음)','R² (높을수록 좋음)'],
    [True, True, False]
):
    vals     = [results[m][metric] for m in model_names]
    bars     = ax.bar(model_names, vals, color=colors, width=0.5, edgecolor='white')
    best_idx = int(np.argmin(vals)) if lower else int(np.argmax(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.4f}', ha='center', fontsize=8)

plt.suptitle('회귀 모델별 성능 비교', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. 분류 모델 학습 (부도 예측)

In [ ]:
FEATURES_CLS = [f for f in FEATURES if f != 'loan_status']
TARGET_CLS   = 'loan_status'

Xc       = df[FEATURES_CLS]
yc       = df[TARGET_CLS]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=SEED
)
print(f'분류 학습: {len(Xc_train):,}행  /  테스트: {len(Xc_test):,}행')
print(f'클래스 비율: {yc.value_counts().to_dict()}')

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
clf.fit(Xc_train, yc_train)

yc_pred = clf.predict(Xc_test)
print(classification_report(yc_test, yc_pred, target_names=['정상(0)','부도(1)']))

In [ ]:
cm = confusion_matrix(yc_test, yc_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['예측 정상','예측 부도'],
            yticklabels=['실제 정상','실제 부도'])
ax.set_title('혼동 행렬 (RandomForestClassifier)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. 최종 모델 저장

In [ ]:
gbr_model   = reg_models['그래디언트 부스팅']
RATE_PATH   = os.path.join(MODEL_DIR, 'loan_rate_model.pkl')
STATUS_PATH = os.path.join(MODEL_DIR, 'loan_status_model.pkl')

joblib.dump(gbr_model, RATE_PATH)
joblib.dump(clf,       STATUS_PATH)

print(f'회귀 모델 저장: {RATE_PATH}')
print(f'분류 모델 저장: {STATUS_PATH}')
r = results['그래디언트 부스팅']
print(f'\n채택 회귀 모델(GBR) — RMSE: {r["RMSE"]}  MAE: {r["MAE"]}  R²: {r["R2"]}')